# Generate the result in RQ4

This script generates the results presented in the RQ4 section of the paper. The script reads the data from the `data/rq4-greybox` folder and generates the results presented in the paper.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
import multiprocessing as mp
import re
from scipy.stats import wilcoxon

# Directory where the coverage records are stored
directory = "data/rq4-greybox"

## Helper functions

In [2]:
# Record class
# exec_per_time: the average number of executions per time unit during the
#                fuzzing campaign
# p_emp_min: the minimum value of the empirical discovery probability
# data_done: the coverage records
@dataclass
class Record:
    exec_per_time: float
    p_emp_min: float
    data_done: pd.DataFrame


# Parse the record file and return a Record object
def get_record_obj(data_path):
    data = pd.read_csv(data_path, sep=", ", engine="python")
    data["done"] = data["done"].astype(bool)
    data["time"] = data["time"] / 1000 / 60
    # if #singletons or #sglt_clusts is 0, set it to 1
    data["#singletons"] = data["#singletons"].replace(0, 1)
    data["#singletonsR"] = data["#singletonsR"].replace(0, 1)
    data["#sglt_clusts"] = data["#sglt_clusts"].replace(0, 1)
    data_done = data[data["done"]]
    # cut off the data after 24 hours
    data_done = data_done[data_done["time"] < 24 * 60]
    data_done.loc[:, "#execs"] = data_done["#execs"].astype(int)

    exec_per_time = data_done.iloc[-1]["#execs"] / data_done.iloc[-1]["time"]
    p_emps = data_done["#foundnew"] / data_done["#execs"]
    p_emps_min = np.min(p_emps[p_emps > 0])

    return Record(exec_per_time, p_emps_min, data_done)


# Given the time t_m, get the empirical discovery probability and
# the discovery probability estimated by the Good-Turing estimator (GT), Mean
# Local estimator with Good-Turing (MLG), and Mean Local estimator with our
# dependency-aware method (MLD)
def discovery(
    record_obj: Record, t_m: float, target: str, debug: bool = False
) -> float:
    exec_at_t = int(t_m * record_obj.exec_per_time)
    if exec_at_t == 0:
        return 0
    # find the row where #execs is the closest to exec_at_t
    for i, row in record_obj.data_done.iterrows():
        if row["#execs"] > exec_at_t:
            break
    row = record_obj.data_done.iloc[i - 1]
    assert row["done"]
    if debug:
        print(row)
    if target == "GT":
        ret = row["#singletons"] / exec_at_t
    elif target == "Reset":
        ret = row["#singletonsR"] / exec_at_t
    elif target == "MLG":
        ret = row["ML_sglt"] / exec_at_t
    elif target == "MLD":
        ret = row["ML_sglt_clusts"] / exec_at_t
    elif target == "emp":
        ret = row["#foundnew"] / exec_at_t
        if ret == 0:
            ret = record_obj.p_emp_min
    else:
        raise ValueError(f"Invalid target: {target}")
    return ret


# Get the analyzable format of the estimation results
def get_plot_df(record_obj: Record, x_scale: str, id: str) -> pd.DataFrame:
    max_time = 24 * 60
    print(f"{id=}, {max_time=}", flush=True)
    if x_scale == "lin":
        x = np.linspace(0, max_time, 100)
    elif x_scale == "log":
        x = np.logspace(0, np.log10(max_time), 100)
    else:
        raise ValueError(f"Invalid x_scale: {x_scale}")
    # limit x to 24 hours
    x = x[x <= min(24 * 60, record_obj.data_done["time"].max())]
    print(f"{id=} processing start", flush=True, end="\r")
    y_emp = [discovery(record_obj, t, "emp") for t in x]
    print(f"{id=} emp done", flush=True, end="\r")
    y_GT = [discovery(record_obj, t, "GT") for t in x]
    print(f"{id=} GT done", flush=True, end="\r")
    y_Reset = [discovery(record_obj, t, "Reset") for t in x]
    print(f"{id=} Reset done", flush=True, end="\r")
    y_MLG = [discovery(record_obj, t, "MLG") for t in x]
    print(f"{id=} MLG done", flush=True, end="\r")
    y_MLD = [discovery(record_obj, t, "MLD") for t in x]
    print(f"{id=} MLD done", flush=True, end="\r")
    plot_df = pd.DataFrame(
        {
            "time": x,
            "p_emp": y_emp,
            "p_GT": y_GT,
            "p_Reset": y_Reset,
            "p_MLG": y_MLG,
            "p_MLD": y_MLD,
        }
    )
    plot_df = plot_df.melt(
        id_vars=["time"],
        value_vars=["p_emp", "p_GT", "p_Reset", "p_MLG", "p_MLD"],
        value_name="Pr",
        var_name="Esti",
    )
    print(f"{id=} done", flush=True)
    plot_df["id"] = id
    return plot_df

In [3]:
# Read fuzzer_stats.csv files, which contains the summary of the
# fuzzing campaign, to get the repetition information
def read_fuzzer_stats(directory):
    """Reads the fuzzer_stats.csv file from the given directory."""
    file_path = os.path.join(directory, "fuzzer_stats.csv")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path)
    return df


def count_indices_for_subject(df, subject):
    """Returns the number of unique indices for a given subject."""
    if "subject" not in df.columns or "index" not in df.columns:
        raise ValueError(
            "DataFrame must contain 'subject' and 'index' columns."
        )

    return sorted(
        [int(i) for i in df[df["subject"] == subject]["index"].unique()]
    )

## Parse the record to generate the data for analysis

In [4]:
get_record_obj(os.path.join(directory, "records/freetype2_0_records.csv"))
datas = [
    data
    for data in os.listdir(os.path.join(directory, "records"))
    if data.endswith("_records.csv")
]
subjects = np.unique([data.split("_")[0] for data in datas]).tolist()
print(f"subjects: {subjects}")

subjects: ['freetype2', 'jsoncpp', 'libjpeg', 'libpcap', 'libpng', 'libxml2', 'sqlite3', 'zlib']


In [5]:
df_fuzzer_stats = read_fuzzer_stats(directory)

for subject_name in subjects:
    index_count = count_indices_for_subject(df_fuzzer_stats, subject_name)
    print(
        f"Number of unique indices for subject '{subject_name}': {index_count}"
    )

Number of unique indices for subject 'freetype2': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Number of unique indices for subject 'jsoncpp': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Number of unique indices for subject 'libjpeg': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Number of unique indices for subject 'libpcap': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Number of unique indices for subject 'libpng': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Number of unique indices for subject 'libxml2': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Number of unique indices for subject 'sqlite3': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Number of unique indices for subject 'zlib': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [6]:
# Generate data for each record and store it in the data directory
# the data is stored in the format of {subject}.csv
# If the data already exists, you can skip this step by setting is_generate
# to False.

is_generate = True
for subject in subjects:
    if is_generate or not os.path.exists(
        os.path.join(directory, f"records/{subject}.csv")
    ):
        print(f"Generating data for {subject}")
        idcs = count_indices_for_subject(df_fuzzer_stats, subject)
        print(
            [
                os.path.join(directory, f"records/{subject}_{i}_records.csv")
                for i in idcs
            ]
        )
        
        # with mp.Pool(len(idcs)) as pool:
        #     record_objs = pool.map(
        #         get_record_obj,
        #         [
        #             os.path.join(
        #                 directory, f"records/{subject}_{i}_records.csv"
        #             )
        #             for i in idcs
        #         ],
        #     )
        record_objs = []
        for i in idcs:
            record_obj = get_record_obj(
                os.path.join(directory, f"records/{subject}_{i}_records.csv")
            )
            record_objs.append(record_obj)
        # params = [
        #     (record_obj, "log", str(idx))
        #     for idx, record_obj in enumerate(record_objs, 1)
        # ]
        # with mp.Pool(len(params)) as pool:
        #     plot_dfs = pool.starmap(get_plot_df, params)
        plot_dfs = []
        for idx, record_obj in enumerate(record_objs, 1):
            plot_df = get_plot_df(record_obj, "log", str(idx))
            plot_dfs.append(plot_df)
            print(f"Processed {idx}/{len(record_objs)}", flush=True, end="\r")
        plot_df = pd.concat(plot_dfs, ignore_index=True)
        plot_df["Esti"] = plot_df["Esti"].replace(
            {
                "p_emp": r"$\hat{m}_{emp}$",
                "p_Reset": r"$\hat{T}_{Reset}$",
                "p_GT": r"$\hat{m}_{GT}$",
                "p_MLG": r"$\hat{m}_{MLG}$",
                "p_MLD": r"$\hat{m}_{MLD}$",
            }
        )
        # save plot_df
        plot_df.to_csv(
            os.path.join(directory, "records/", f"{subject}.csv"), index=False
        )

Generating data for freetype2
['data/rq4-greybox/records/freetype2_0_records.csv', 'data/rq4-greybox/records/freetype2_1_records.csv', 'data/rq4-greybox/records/freetype2_2_records.csv', 'data/rq4-greybox/records/freetype2_3_records.csv', 'data/rq4-greybox/records/freetype2_4_records.csv', 'data/rq4-greybox/records/freetype2_5_records.csv', 'data/rq4-greybox/records/freetype2_6_records.csv', 'data/rq4-greybox/records/freetype2_7_records.csv', 'data/rq4-greybox/records/freetype2_8_records.csv', 'data/rq4-greybox/records/freetype2_9_records.csv', 'data/rq4-greybox/records/freetype2_10_records.csv', 'data/rq4-greybox/records/freetype2_11_records.csv', 'data/rq4-greybox/records/freetype2_12_records.csv', 'data/rq4-greybox/records/freetype2_13_records.csv', 'data/rq4-greybox/records/freetype2_14_records.csv', 'data/rq4-greybox/records/freetype2_15_records.csv', 'data/rq4-greybox/records/freetype2_16_records.csv', 'data/rq4-greybox/records/freetype2_17_records.csv', 'data/rq4-greybox/records

## Discovery probability estimation accuracy (RQ4)

It generates
- Overall discovery probability estimation accuracy (Table 5 in the main paper)
- Discovery probability estimation accuracy at each time interval of the greybox fuzzing (Table 3 in the supplementary material)

In [7]:
# Aggregate the data
data = []
for subject in subjects:
    plot_df = pd.read_csv(os.path.join(directory, f"records/{subject}.csv"))
    sub_df = plot_df[["id", "time", "Esti", "Pr"]]
    for id in sub_df["id"].unique():
        sub_sub_df = sub_df[sub_df["id"] == id].copy()
        sub_sub_df["Esti"] = sub_sub_df["Esti"].replace(
            {
                r"$\hat{m}_{emp}$": "emp",
                r"$\hat{m}_{GT}$": "GT",
                r"$\hat{T}_{Reset}$": "Reset",
                r"$\hat{m}_{MLG}$": "MLG",
                r"$\hat{m}_{MLD}$": "MLD",
            }
        )
        sub_sub_df = (
            sub_sub_df.pivot(index="time", columns="Esti", values="Pr")
            .reset_index()
            .rename_axis(None, axis=1)
        )
        logerr_gt = np.log10(sub_sub_df["GT"]) - np.log10(sub_sub_df["emp"])
        logerr_reset = np.log10(sub_sub_df["Reset"]) - np.log10(sub_sub_df["emp"])
        logerr_mlg = np.log10(sub_sub_df["MLG"]) - np.log10(sub_sub_df["emp"])
        logerr_mld = np.log10(sub_sub_df["MLD"]) - np.log10(sub_sub_df["emp"])
        err_gt = np.abs(sub_sub_df["GT"] - sub_sub_df["emp"])
        err_reset = np.abs(sub_sub_df["Reset"] - sub_sub_df["emp"])
        err_mlg = np.abs(sub_sub_df["MLG"] - sub_sub_df["emp"])
        err_mld = np.abs(sub_sub_df["MLD"] - sub_sub_df["emp"])
        rel_err_gt = err_gt / sub_sub_df["emp"]
        rel_err_reset = err_reset / sub_sub_df["emp"]
        rel_err_mlg = err_mlg / sub_sub_df["emp"]
        rel_err_mld = err_mld / sub_sub_df["emp"]

        time_intervals = [(1, 10), (10, 60), (60, 360), (360, 1440), (1, 1440)]
        for start_time, end_time in time_intervals:
            time_id = f"{start_time}-{end_time}"
            sub3_df = sub_sub_df[
                (sub_sub_df["time"] >= start_time)
                & (sub_sub_df["time"] <= end_time)
            ]
            time_diff = sub3_df["time"].diff().dropna()
            mid = lambda x: (x.iloc[1:] + x.iloc[:-1]) / 2
            weighted_avg = lambda x: np.sum(x.iloc[1:] * time_diff) / np.sum(
                time_diff
            )
            emp_avg = weighted_avg(mid(sub3_df["emp"]))
            logerr_gt_avg = weighted_avg(mid(logerr_gt))
            logerr_reset_avg = weighted_avg(mid(logerr_reset))
            logerr_mlg_avg = weighted_avg(mid(logerr_mlg))
            logerr_mld_avg = weighted_avg(mid(logerr_mld))
            err_gt_avg = weighted_avg(mid(err_gt))
            err_reset_avg = weighted_avg(mid(err_reset))
            err_mlg_avg = weighted_avg(mid(err_mlg))
            err_mld_avg = weighted_avg(mid(err_mld))
            rel_err_gt_avg = weighted_avg(mid(rel_err_gt))
            rel_err_reset_avg = weighted_avg(mid(rel_err_reset))
            rel_err_mlg_avg = weighted_avg(mid(rel_err_mlg))
            rel_err_mld_avg = weighted_avg(mid(rel_err_mld))
            data.append(
                {
                    "subject": subject,
                    "id": id,
                    "interval": time_id,
                    "emp_avg": emp_avg,
                    "logerr_gt_avg": logerr_gt_avg,
                    "logerr_reset_avg": logerr_reset_avg,
                    "logerr_mlg_avg": logerr_mlg_avg,
                    "logerr_mld_avg": logerr_mld_avg,
                    "err_gt_avg": err_gt_avg,
                    "err_reset_avg": err_reset_avg,
                    "err_mlg_avg": err_mlg_avg,
                    "err_mld_avg": err_mld_avg,
                    "rel_err_gt_avg": rel_err_gt_avg,
                    "rel_err_reset_avg": rel_err_reset_avg,
                    "rel_err_mlg_avg": rel_err_mlg_avg,
                    "rel_err_mld_avg": rel_err_mld_avg,
                }
            )
data_df = pd.DataFrame(data)
# assign very small value to 0 values
data_df.replace(0, 1e-20, inplace=True)
data_df

,subject,id,interval,emp_avg,logerr_gt_avg,logerr_reset_avg,logerr_mlg_avg,logerr_mld_avg,err_gt_avg,err_reset_avg,err_mlg_avg,err_mld_avg,rel_err_gt_avg,rel_err_reset_avg,rel_err_mlg_avg,rel_err_mld_avg
0,freetype2,1,1-10,0.029908,-2.454668,-0.392562,2.822175,1.206735,0.030788,0.018916,19.620583,0.423531,0.988138,0.621126,700.554382,16.509094
1,freetype2,1,10-60,0.021389,-3.830759,-1.095778,2.332250,1.246472,0.027915,0.025716,5.210401,0.384240,0.999726,0.880314,272.990698,19.206613
2,freetype2,1,60-360,0.041667,-4.969216,-2.191711,1.700379,0.918706,0.046730,0.046382,2.329592,0.319775,0.999988,0.992137,53.620807,7.621693
3,freetype2,1,360-1440,0.027819,-4.850379,-2.781497,1.356808,0.963907,0.027819,0.027771,0.815295,0.265475,0.900683,0.899169,29.441400,10.501348
4,freetype2,1,1-1440,0.032890,-4.835904,-2.573915,1.475831,0.960121,0.032886,0.032613,1.445350,0.283180,0.929029,0.919424,47.752995,10.070122
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,zlib,20,1-10,0.028167,-4.015406,-3.786098,1.369329,1.283182,0.030152,0.030149,0.295732,0.221344,0.980238,1.061897,1669.515331,1218.565130
796,zlib,20,10-60,0.003269,-2.761470,-2.704666,3.355171,3.349755,0.003270,0.003270,0.247587,0.244454,0.985330,0.979865,23996.150176,23806.548657
797,zlib,20,60-360,0.005101,-3.644196,-3.644196,3.241720,3.240872,0.005101,0.005101,0.234080,0.233612,0.995719,0.995719,26242.636132,26187.213367
798,zlib,20,360-1440,0.001439,-4.183287,-4.183287,2.559038,2.558941,0.001439,0.001439,0.208126,0.208073,0.900066,0.900066,12397.480272,12391.975422


### Overall discovery probability estimation accuracy (Table 5 in the main paper)

In [8]:
# average
data_df_avg = (
    data_df.replace([np.inf, -np.inf], np.nan)
    .dropna()
    .groupby(["subject", "interval"])
    .mean()
    .reset_index()
)
data_df_avg["ratio"] = data_df_avg["err_mld_avg"] / data_df_avg["err_mlg_avg"]
# subject order: "sqlite3", "freetype2", "libxml2", "libjpeg", "zlib",
# "libpcap", "jsoncpp", "libpng"
data_df_avg["subject"] = pd.Categorical(
    data_df_avg["subject"],
    categories=[
        "sqlite3",
        "freetype2",
        "libxml2",
        "libjpeg",
        "zlib",
        "libpcap",
        "jsoncpp",
        "libpng",
    ],
    ordered=True,
)
# interval order: ["1-10", "10-60", "60-360", "360-1440", "1-1440"]
data_df_avg["interval"] = pd.Categorical(
    data_df_avg["interval"],
    categories=["1-10", "10-60", "60-360", "360-1440", "1-1440"],
    ordered=True,
)
data_df_avg = data_df_avg.sort_values(["subject", "interval"])

In [9]:
total_df_avg = data_df_avg[data_df_avg["interval"] == "1-1440"]
total_df_avg = total_df_avg[
    [
        "subject",
        "emp_avg",
        "logerr_reset_avg",
        "logerr_mlg_avg",
        "logerr_mld_avg",
        "err_reset_avg",
        "err_mlg_avg",
        "err_mld_avg",
        "rel_err_reset_avg",
        "rel_err_mlg_avg",
        "rel_err_mld_avg",
        "ratio",
    ]
]
# change the name of the columns
total_df_avg.columns = [
    "Subject",
    r"$\hat{m}_{\mathit{emp}}$",
    r"$\mathit{LE}_{\mathit{Reset}}$",
    r"$\mathit{LE}_{\mathit{MLG}}$",
    r"$\mathit{LE}_{\mathit{MLD}}$",
    r"$\mathit{AE}_{\mathit{Reset}}$",
    r"$\mathit{AE}_{\mathit{MLG}}$",
    r"$\mathit{AE}_{\mathit{MLD}}$",
    r"$\mathit{RE}_{\mathit{Reset}}$",
    r"$\mathit{RE}_{\mathit{MLG}}$",
    r"$\mathit{RE}_{\mathit{MLD}}$",
    r"$\frac{\mathit{AE}_{\mathit{MLD}}}{\mathit{AE}_{\mathit{MLG}}}$",
]
total_df_avg = total_df_avg.reset_index(drop=True)
total_df_avg = total_df_avg.astype(str)
total_df_avg["Subject"] = total_df_avg["Subject"].map(
    lambda x: "\\" + x
)
total_df_avg["Subject"] = total_df_avg["Subject"].map(
    lambda x: re.sub(r"\d+$", "", x)
)
total_df_avg.iloc[:, 1] = total_df_avg.iloc[:, 1].map(
    lambda x: f"{float(x):.2e}"
)
total_df_avg.iloc[:, 2:5] = total_df_avg.iloc[:, 2:5].map(
    lambda x: f"{float(x):.2f}"
)
total_df_avg.iloc[:, 5:8] = total_df_avg.iloc[:, 5:8].map(
    lambda x: f"{float(x):.2e}"
)
total_df_avg.iloc[:, 8:12] = total_df_avg.iloc[:, 8:12].map(
    lambda x: f"{float(x):.2f}"
)
total_df_avg = total_df_avg.reset_index(drop=True)
display(total_df_avg)

,Subject,$\hat{m}_{\mathit{emp}}$,$\mathit{LE}_{\mathit{Reset}}$,$\mathit{LE}_{\mathit{MLG}}$,$\mathit{LE}_{\mathit{MLD}}$,$\mathit{AE}_{\mathit{Reset}}$,$\mathit{AE}_{\mathit{MLG}}$,$\mathit{AE}_{\mathit{MLD}}$,$\mathit{RE}_{\mathit{Reset}}$,$\mathit{RE}_{\mathit{MLG}}$,$\mathit{RE}_{\mathit{MLD}}$,$\frac{\mathit{AE}_{\mathit{MLD}}}{\mathit{AE}_{\mathit{MLG}}}$
0,\sqlite,9.57e-02,-2.26,1.44,0.36,9.49e-02,4.60e+00,1.34e-01,0.92,45.95,1.35,0.03
1,\freetype,3.07e-02,-2.49,1.48,0.98,3.04e-02,1.43e+00,2.87e-01,0.92,49.50,10.54,0.20
2,\libxml,1.85e-02,-2.01,1.88,1.18,1.80e-02,2.60e+00,3.17e-01,0.91,120.25,16.63,0.12
3,\libjpeg,1.24e-02,-3.87,1.58,1.34,1.23e-02,3.53e-01,1.60e-01,0.93,350.16,245.86,0.45
4,\zlib,3.58e-03,-3.93,2.78,2.78,3.58e-03,2.17e-01,2.17e-01,0.93,7364.87,7354.14,1.00
5,\libpcap,1.95e-02,-5.29,1.10,1.06,1.95e-02,2.22e-01,1.92e-01,0.93,17.20,15.42,0.87
6,\jsoncpp,1.12e-03,-3.09,2.87,2.83,1.12e-03,2.82e-01,2.18e-01,0.93,3570.90,3258.92,0.77
7,\libpng,6.59e-03,-4.46,1.65,1.62,6.58e-03,2.58e-01,2.27e-01,0.93,454.99,390.82,0.88


### Discovery probability estimation accuracy at each time interval of the greybox fuzzing (Table 3 in the supplementary material)

In [10]:
data_df_sup = data_df_avg[~data_df_avg["interval"].isin(["1-1440"])]
data_df_sup = data_df_sup[
    [
        "subject",
        "interval",
        "emp_avg",
        "logerr_reset_avg",
        "logerr_mlg_avg",
        "logerr_mld_avg",
        "err_reset_avg",
        "err_mlg_avg",
        "err_mld_avg",
        "rel_err_reset_avg",
        "rel_err_mlg_avg",
        "rel_err_mld_avg",
        "ratio",
    ]
]
data_df_sup["interval"] = pd.Categorical(
    data_df_sup["interval"],
    categories=["1-10", "10-60", "60-360", "360-1440"],
    ordered=True,
)
data_df_sup = data_df_sup.sort_values(["subject", "interval"])
# change the name of the columns
data_df_sup.columns = [
    "Subject",
    "Interval (min)",
    r"$\mu(\hat{m}_{\mathit{emp}})$",
    r"$\mathit{LE}_{\mathit{Reset}}$",
    r"$\mathit{LE}_{\mathit{MLG}}$",
    r"$\mathit{LE}_{\mathit{MLD}}$",
    r"$\mathit{AE}_{\mathit{Reset}}$",
    r"$\mathit{AE}_{\mathit{MLG}}$",
    r"$\mathit{AE}_{\mathit{MLD}}$",
    r"$\mathit{RE}_{\mathit{Reset}}$",
    r"$\mathit{RE}_{\mathit{MLG}}$",
    r"$\mathit{RE}_{\mathit{MLD}}$",
    r"$\frac{\mathit{AE}_{\mathit{MLD}}}{\mathit{AE}_{\mathit{MLG}}}$",
]
data_df_sup = data_df_sup.reset_index(drop=True)
data_df_sup = data_df_sup.astype(str)
data_df_sup["Subject"] = data_df_sup["Subject"].map(
    lambda x: "\\" + x
)
data_df_sup["Subject"] = data_df_sup["Subject"].map(
    lambda x: re.sub(r"\d+$", "", x)
)
data_df_sup.iloc[:, 2] = data_df_sup.iloc[:, 2].map(
    lambda x: f"{float(x):.2e}"
)
data_df_sup.iloc[:, 3:6] = data_df_sup.iloc[:, 3:6].map(
    lambda x: f"{float(x):.2f}"
)
data_df_sup.iloc[:, 6:9] = data_df_sup.iloc[:, 6:9].map(
    lambda x: f"{float(x):.2e}"
)
data_df_sup.iloc[:, 9:13] = data_df_sup.iloc[:, 9:13].map(
    lambda x: f"{float(x):.2f}"
)
data_df_sup = data_df_sup.reset_index(drop=True)
display(data_df_sup)

,Subject,Interval (min),$\mu(\hat{m}_{\mathit{emp}})$,$\mathit{LE}_{\mathit{Reset}}$,$\mathit{LE}_{\mathit{MLG}}$,$\mathit{LE}_{\mathit{MLD}}$,$\mathit{AE}_{\mathit{Reset}}$,$\mathit{AE}_{\mathit{MLG}}$,$\mathit{AE}_{\mathit{MLD}}$,$\mathit{RE}_{\mathit{Reset}}$,$\mathit{RE}_{\mathit{MLG}}$,$\mathit{RE}_{\mathit{MLD}}$,$\frac{\mathit{AE}_{\mathit{MLD}}}{\mathit{AE}_{\mathit{MLG}}}$
0,\sqlite,1-10,8.00e-02,-0.40,2.80,0.73,8.85e-02,8.32e+01,3.78e-01,1.09,1075.39,4.69,0.00
1,\sqlite,10-60,1.13e-01,-1.37,2.06,0.38,1.17e-01,1.53e+01,1.71e-01,0.94,129.58,1.46,0.01
2,\sqlite,60-360,1.01e-01,-2.02,1.72,0.36,1.09e-01,6.39e+00,1.42e-01,0.99,58.65,1.33,0.02
3,\sqlite,360-1440,8.97e-02,-2.40,1.31,0.35,8.94e-02,2.75e+00,1.26e-01,0.90,27.95,1.31,0.05
4,\freetype,1-10,3.24e-02,-0.35,2.79,1.16,2.31e-02,2.00e+01,4.15e-01,0.63,706.84,14.84,0.02
5,\freetype,10-60,2.97e-02,-1.23,2.32,1.14,3.09e-02,6.72e+00,3.87e-01,0.92,242.11,14.05,0.06
6,\freetype,60-360,3.60e-02,-2.16,1.75,1.01,3.87e-02,2.14e+00,3.29e-01,0.99,63.70,10.19,0.15
7,\freetype,360-1440,2.78e-02,-2.68,1.33,0.96,2.78e-02,7.64e-01,2.66e-01,0.90,29.18,10.43,0.35
8,\libxml,1-10,3.88e-02,-0.26,2.97,1.10,2.00e-02,4.00e+01,4.69e-01,0.48,991.76,12.09,0.01
9,\libxml,10-60,2.58e-02,-0.91,2.62,1.24,2.40e-02,1.22e+01,4.42e-01,0.86,436.51,16.59,0.04


### Statistical analysis

In [11]:
results = []

for (subject, interval), group in data_df.groupby(['subject', 'interval']):
    mlg = group['err_mlg_avg'].values
    mld = group['err_mld_avg'].values
    try:
        stat, p_value = wilcoxon(
            mlg, mld, alternative='two-sided'
        )
    except ValueError:  # Handle cases where values are identical
        stat, p_value = np.nan, np.nan

    # Compute effect size
    delta = 1 - 2 * stat / (len(mlg) * (len(mlg) + 1))
    magnitude = ""
    if delta < 0.1:
        magnitude = "negligible"
    elif delta < 0.3:
        magnitude = "small"
    elif delta < 0.5:
        magnitude = "medium"
    else:
        magnitude = "large"

    results.append({
        'subject': subject,
        'interval': interval,
        'mean_mlg': mlg.mean(),
        'mean_mld': mld.mean(),
        'p_value': p_value,
        'delta': delta,
    })

results_df = pd.DataFrame(results)
display(results_df)

,subject,interval,mean_mlg,mean_mld,p_value,delta
0,freetype2,1-10,20.008106,0.415080,0.000002,1.0
1,freetype2,1-1440,1.430640,0.286541,0.000002,1.0
2,freetype2,10-60,6.722992,0.386960,0.000002,1.0
3,freetype2,360-1440,0.764208,0.266205,0.000002,1.0
4,freetype2,60-360,2.143107,0.328964,0.000002,1.0
5,jsoncpp,1-10,3.583929,0.319312,0.000002,1.0
6,jsoncpp,1-1440,0.281722,0.217580,0.000002,1.0
7,jsoncpp,10-60,0.924818,0.278118,0.000002,1.0
8,jsoncpp,360-1440,0.213620,0.205819,0.000002,1.0
9,jsoncpp,60-360,0.300559,0.240788,0.000002,1.0
